# A3 ABO · 04 Analysis

Kiểm lại kết quả từ dự đoán theo ID và tạo bảng/biểu đồ cho báo cáo. Có thể chạy từ thư mục suite đã sao chép mà không cần encoder cache.

Colab dùng A3_ABO_team.zip trong MyDrive/A3_ABO. Jupyter local chạy từ checkout A3 hoặc đặt biến A3_REPO. Dùng cùng A3_WORK, tên cohort và A3_RUN_ID cho các notebook.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
import json
import os
import shutil
import subprocess
import sys

ON_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# Edit paths here if using another Drive folder or a checkout outside the current directory.
WORK = Path(os.environ.get("A3_WORK", "/content/drive/MyDrive/A3_ABO" if ON_COLAB else "runs/a3/team")).resolve()
REPO = Path(os.environ.get("A3_REPO", ".")).resolve()
if not (REPO / "src/dlbench/a3/curation.py").is_file():
    if not ON_COLAB and (REPO.parent / "src/dlbench/a3/curation.py").is_file():
        REPO = REPO.parent
    elif ON_COLAB:
        bundle = WORK / "A3_ABO_team.zip"
        if bundle.is_file():
            digest = hashlib.sha256(bundle.read_bytes()).hexdigest()[:16]
            REPO = Path("/content") / f"a3-source-{digest}"
            if not (REPO / "src/dlbench/a3/curation.py").is_file():
                shutil.unpack_archive(bundle, REPO)
        elif (WORK / "setup.py").is_file() and (WORK / "src/dlbench/a3/team_batches.py").is_file():
            REPO = WORK
        else:
            raise FileNotFoundError(f"Đặt A3_ABO_team.zip trong WORK hoặc chỉnh WORK. Đang dùng: {WORK}")
    else:
        raise FileNotFoundError("Set A3_REPO to the A3 checkout or extracted source ZIP.")

if ON_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[a3]"], check=True)
source = str(REPO / "src")
if source not in sys.path:
    sys.path.insert(0, source)
env = {**os.environ, "PYTHONPATH": source + os.pathsep + os.environ.get("PYTHONPATH", "")}

def run_a3(*args):
    subprocess.run([sys.executable, "-m", "dlbench.a3.cli", *map(str, args)],
                   cwd=REPO, env=env, check=True)

WORK.mkdir(parents=True, exist_ok=True)
DATA = WORK / "data"
IMAGE_ROOT = DATA / "images"
FROZEN = WORK / "cohorts" / "abo-v1"
IMAGE_CACHE = FROZEN / "image.npz"
TITLE_CACHE = FROZEN / "title.npz"
MANIFEST = FROZEN / "manifest.csv"
RUN_ID = os.environ.get("A3_RUN_ID", "abo-v1-main")
SUITE = WORK / "runs" / RUN_ID
INPUT_ARGS = ("--manifest", MANIFEST, "--image-cache", IMAGE_CACHE, "--title-cache", TITLE_CACHE)
print(f"Source: {REPO}\nWorkspace: {WORK}")

In [ ]:
from IPython.display import Image as DisplayImage, display

OUTPUT = SUITE / "analysis"
if not (OUTPUT / "summary.json").is_file():
    run_a3("analyze", "--suite", SUITE, "--output", OUTPUT)
summary = json.loads((OUTPUT / "summary.json").read_text(encoding="utf-8"))
for model, values in summary["by_model"].items():
    print(f"{model:15} macro-F1 {values['test_macro_f1']['mean']:.3%} | "
          f"accuracy {values['test_accuracy']['mean']:.3%}")
for name, values in summary["paired"].items():
    print(f"{name:26} {values['mean_delta'] * 100:+.2f} điểm % macro-F1")
for figure in ("test_macro_f1.png", "validation_curves.png", "test_confusions.png"):
    display(DisplayImage(filename=str(OUTPUT / figure)))
print(OUTPUT)

In [ ]:
import csv

COMPARISON = "product_vs_concat"
with (OUTPUT / "error_cases.csv").open(encoding="utf-8", newline="") as stream:
    cases = list(csv.DictReader(stream))
for change in ("corrected", "introduced_error"):
    selected = [row for row in cases if row["comparison"] == COMPARISON and row["change"] == change]
    print(change, len(selected))
    for row in selected[:3]:
        print(row["sample_id"], row["seed"], row["label"], row["product"], row["concat"], row["title"])
        path = IMAGE_ROOT / row["image_path"]
        if path.is_file():
            from PIL import Image
            with Image.open(path) as photo:
                thumb = photo.copy()
                thumb.thumbnail((180, 180))
                display(thumb)